# SFT + GRPO v1 Fine-Tuning & Evaluation: `google/translategemma-4b-it`
## Spanish → Valencian (ES → VA)

**Pipeline:**
1. Install dependencies
2. Load `google/translategemma-4b-it` with 4-bit quantisation + LoRA
3. **SFT** on `gplsi/amic_parallel` (50k ES→VA pairs)
4. **GRPO v1** — composite reward: chrF + P(HT|text) classifier, warm-up schedule
5. **Evaluation** on `gplsi/ES-VA_translation_test` (1k sentences)

| Model | HuggingFace Hub |
|---|---|
| SFT | `guerreropaula/translategemma4b-sft-es-va2` |
| GRPO v1 | `guerreropaula/80translategemma4b-grpo-es-va` |
| HT/MT Classifier | `guerreropaula/ht_mt_classifier_best` |


---
## 1. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers>=4.49.0 datasets>=2.18.0 accelerate>=0.27.0 \
    peft>=0.9.0 trl>=0.8.6 bitsandbytes>=0.42.0 sacrebleu>=2.4.0 \
    sentencepiece>=0.1.99 huggingface_hub>=0.22.0
!pip install -q git+https://github.com/google-research/bleurt.git
!wget -q https://storage.googleapis.com/bleurt-oss/bleurt-base-128.zip && unzip -q bleurt-base-128.zip

In [ ]:
import torch, gc
import transformers, peft, trl
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | trl {trl.__version__}")
print(f"CUDA: {torch.cuda.is_available()}", f"| GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")

---
## 2. Configuration

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # <- set your HuggingFace token here, or leave empty for interactive login
login(token=HF_TOKEN) if HF_TOKEN else login()

BASE_MODEL_ID = "google/translategemma-4b-it"
SFT_HUB_ID    = "guerreropaula/translategemma4b-sft-es-va2"
GRPO_HUB_ID   = "guerreropaula/80translategemma4b-grpo-es-va"
CLF_REPO_ID   = "guerreropaula/ht_mt_classifier_best"

SOURCE_LANG_CODE = "es"
TARGET_LANG_CODE = "ca"   # TranslateGemma does not distinguish VA from CA
SOURCE_COL, TARGET_COL = "ES", "VA"

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported()
print(f"Device: {DEVICE} | BF16: {USE_BF16}")

---
## 3. Load Base Model (4-bit NF4 + LoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MAX_SEQ_LENGTH = 256

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN,
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

---
## 4. TranslateGemma Prompt Template

In [ ]:
def _make_messages(source_text):
    return [{"role":"user","content":[{
        "type":"text","source_lang_code":SOURCE_LANG_CODE,
        "target_lang_code":TARGET_LANG_CODE,"text":source_text
    }]}]

def format_for_sft(source_text, target_text):
    prompt = tokenizer.apply_chat_template(_make_messages(source_text), tokenize=False, add_generation_prompt=True)
    return prompt + target_text + tokenizer.eos_token

def make_inference_prompt(source_text):
    return tokenizer.apply_chat_template(_make_messages(source_text), tokenize=False, add_generation_prompt=True)

print(make_inference_prompt("El ayuntamiento aprobó el presupuesto."))

---
## 5. Load SFT Dataset

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("gplsi/amic_parallel")
print(raw_dataset)

---
## 6. SFT Training

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForLanguageModeling, TrainerCallback
import matplotlib.pyplot as plt

SFT_TRAIN_SAMPLES = 50_000
SFT_OUTPUT_DIR    = "./translategemma4b_sft"

def formatting_prompts_func(examples):
    return {"text": [format_for_sft(s, t) for s, t in zip(examples[SOURCE_COL], examples[TARGET_COL])]}

sft_dataset = raw_dataset.map(formatting_prompts_func, batched=True, remove_columns=raw_dataset["train"].column_names)

class Gemma3DataCollator:
    """Injects token_type_ids (zeros) required by Gemma 3's attention mask."""
    def __init__(self, tok):
        self.base = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
    def __call__(self, features):
        batch = self.base(features)
        batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])
        return batch

class LossPlotCallback(TrainerCallback):
    def __init__(self): self.steps, self.losses = [], []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.steps.append(state.global_step); self.losses.append(logs["loss"])
            plt.figure(figsize=(10,3)); plt.plot(self.steps, self.losses, lw=1.5)
            plt.title("SFT Loss"); plt.grid(alpha=0.3); plt.tight_layout()
            plt.savefig("sft_loss.png", dpi=150); plt.close()

torch.cuda.empty_cache(); gc.collect()
model.train()
sft_trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=sft_dataset["train"].shuffle(seed=42).select(range(SFT_TRAIN_SAMPLES)),
    data_collator=Gemma3DataCollator(tokenizer),
    callbacks=[LossPlotCallback()],
    args=SFTConfig(
        packing=False, per_device_train_batch_size=1, gradient_accumulation_steps=32,
        warmup_steps=25, max_steps=2000, learning_rate=2e-4, logging_steps=25,
        optim="paged_adamw_8bit", weight_decay=0.001, lr_scheduler_type="cosine",
        fp16=not USE_BF16, bf16=USE_BF16, gradient_checkpointing=True,
        max_seq_length=MAX_SEQ_LENGTH, output_dir=SFT_OUTPUT_DIR,
        save_steps=500, seed=42, report_to="none",
    ),
)
sft_stats = sft_trainer.train()
print(f"SFT done | loss={sft_stats.metrics.get('train_loss',0):.4f} | time={sft_stats.metrics['train_runtime']:.0f}s")

---
## 7. Save & Push SFT

In [ ]:
model.save_pretrained(SFT_OUTPUT_DIR+"/lora_adapter")
tokenizer.save_pretrained(SFT_OUTPUT_DIR+"/lora_adapter")
model.push_to_hub(SFT_HUB_ID, token=HF_TOKEN)
tokenizer.push_to_hub(SFT_HUB_ID, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{SFT_HUB_ID}")

---
## 8. GRPO v1 Reward Setup

**Reward:** `r_final = (1 − α) · r_c + α · r_t`
- `r_c` = sentence chrF/100 (reference fidelity)
- `r_t` = P(HT|text) from translationese classifier
- `α` = 0 → 0.3 linearly after 50 warm-up steps


In [ ]:
import sacrebleu
import torch.nn.functional as F
from typing import List
from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification as _AMSC

_clf_tok   = _AT.from_pretrained(CLF_REPO_ID)
_clf_model = _AMSC.from_pretrained(CLF_REPO_ID).eval().to(DEVICE)
print(f"Classifier: {CLF_REPO_ID} | Labels: {_clf_model.config.id2label}")

CLF_WARMUP_STEPS = 50
CLF_WEIGHT_MAX   = 0.3
TOTAL_STEPS      = 200
_step = {"n": 0}

def _clf_alpha():
    s = _step["n"]
    if s < CLF_WARMUP_STEPS: return 0.0
    return CLF_WEIGHT_MAX * min(1.0, (s-CLF_WARMUP_STEPS)/max(1,TOTAL_STEPS-CLF_WARMUP_STEPS))

@torch.no_grad()
def translationese_reward(texts, batch_size=16):
    rewards = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = _clf_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        probs = F.softmax(_clf_model(**enc).logits, dim=-1)
        rewards.extend(probs[:, 1].cpu().tolist())   # index 1 = HT
    return rewards

def content_reward(hyps, refs):
    return [sacrebleu.sentence_chrf(h,[r]).score/100.0 for h,r in zip(hyps,refs)]

def composite_reward(hyps, refs):
    alpha = _clf_alpha()
    rc    = content_reward(hyps, refs)
    if alpha > 0:
        rt = translationese_reward(hyps)
        return [(1-alpha)*c + alpha*t for c,t in zip(rc,rt)]
    return rc

---
## 9. GRPO v1 Training

In [ ]:
GRPO_TRAIN_SAMPLES = 5_000
GRPO_OUTPUT_DIR    = "./translategemma4b_grpo_v1"

def make_grpo_example(examples):
    return {"prompt":[make_inference_prompt(s) for s in examples[SOURCE_COL]],
            "reference":list(examples[TARGET_COL])}

grpo_dataset = raw_dataset["train"].shuffle(seed=123).select(range(GRPO_TRAIN_SAMPLES))
grpo_dataset = grpo_dataset.map(make_grpo_example, batched=True, remove_columns=grpo_dataset.column_names)

In [ ]:
import importlib
import transformers.models.gemma3.modeling_gemma3 as _g3
from trl import GRPOConfig, GRPOTrainer

importlib.reload(_g3)
_g3._orig_mask = _g3.create_causal_mask_mapping
def _patch(config, input_embeds, attention_mask, cache_position, past_key_values,
           position_ids, token_type_ids=None, **kw):
    if token_type_ids is None:
        token_type_ids = torch.zeros(input_embeds.shape[:2], dtype=torch.long, device=input_embeds.device)
    return _g3._orig_mask(config, input_embeds, attention_mask, cache_position,
                          past_key_values, position_ids, token_type_ids=token_type_ids, **kw)
_g3.create_causal_mask_mapping = _patch

def grpo_reward_fn(prompts, completions, reference=None, **kwargs):
    clean = [c.split("model\n")[-1].strip() for c in completions]
    _step["n"] += 1
    rewards = composite_reward(clean, list(reference)) if reference else content_reward(clean, [""]*len(clean))
    print(f"[reward] step={_step['n']:3d} alpha={_clf_alpha():.3f} mean={sum(rewards)/len(rewards):.4f}")
    return rewards

tokenizer.padding_side = "left"
grpo_config = GRPOConfig(
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=5e-6, max_steps=TOTAL_STEPS, warmup_steps=20,
    optim="paged_adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
    gradient_checkpointing=True, beta=0.04, num_generations=2,
    max_completion_length=100, temperature=0.9,
    fp16=not USE_BF16, bf16=USE_BF16,
    output_dir=GRPO_OUTPUT_DIR, logging_steps=1, save_steps=20, seed=3407, report_to="none",
)
grpo_trainer = GRPOTrainer(model=model, processing_class=tokenizer,
                            reward_funcs=grpo_reward_fn, args=grpo_config, train_dataset=grpo_dataset)
model.train()
grpo_stats = grpo_trainer.train()
print(f"GRPO done | time={grpo_stats.metrics['train_runtime']:.0f}s")

---
## 10. Save & Push GRPO v1

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained(GRPO_OUTPUT_DIR+"_merged")
tokenizer.save_pretrained(GRPO_OUTPUT_DIR+"_merged")
merged.push_to_hub(GRPO_HUB_ID, token=HF_TOKEN)
tokenizer.push_to_hub(GRPO_HUB_ID, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{GRPO_HUB_ID}")

---
## 11. Evaluation — Baseline vs SFT vs GRPO v1 vs GRPO v2

Dataset: `gplsi/ES-VA_translation_test` (1,000 longest sentences)  
Metrics: chrF, BLEU, TER, BLEURT


In [ ]:
from datasets import load_dataset as _ld
from bleurt import score as bleurt_score
from peft import PeftModel

bleurt_scorer = bleurt_score.BleurtScorer("bleurt-base-128")

eval_ds     = _ld("gplsi/ES-VA_translation_test", split="test")
eval_sorted = eval_ds.map(lambda x: {"len":len(x["es"])}).sort("len", reverse=True)
EVAL_N      = 1000
eval_raw    = eval_sorted.select(range(EVAL_N))
gold_es     = [ex["es"] for ex in eval_raw]
gold_va     = [ex["va"] for ex in eval_raw]
print(f"Eval set: {EVAL_N} sentences | avg length: {sum(len(s) for s in gold_es)/EVAL_N:.0f} chars")

In [ ]:
@torch.no_grad()
def translate_batch(mdl, tok, sources, batch_size=4):
    mdl.eval(); tok.padding_side = "left"
    hyps, skipped = [], []
    for i in range(0, len(sources), batch_size):
        batch   = sources[i:i+batch_size]
        prompts = [make_inference_prompt(s) for s in batch]
        enc     = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
        try:
            out = mdl.generate(**enc, max_new_tokens=150, do_sample=False, pad_token_id=tok.pad_token_id)
            for o in out:
                t = tok.decode(o, skip_special_tokens=True)
                hyps.append(t.split("model")[-1].strip() if "model" in t else t)
        except Exception as e:
            print(f"  [WARN] {e}"); [hyps.append("[SKIPPED]") for _ in batch]
    return hyps, [i for i,h in enumerate(hyps) if h=="[SKIPPED]"]

def compute_metrics(hyps, refs, skipped, label):
    vh = [h for i,h in enumerate(hyps) if i not in skipped]
    vr = [r for i,r in enumerate(refs)  if i not in skipped]
    bl = bleurt_scorer.score(references=vr, candidates=vh)
    return {"model":label.upper(),
            "chrF":round(sacrebleu.corpus_chrf(vh,[vr]).score,4),
            "BLEU":round(sacrebleu.corpus_bleu(vh,[vr]).score,4),
            "TER" :round(sacrebleu.corpus_ter(vh,[vr]).score,4),
            "BLEURT":round(sum(bl)/len(bl),4) if bl else 0.0,
            "n_eval":len(vh),"skipped":len(skipped)}

def evaluate_model(mdl, tok, label, n=EVAL_N):
    hyps, skip = translate_batch(mdl, tok, gold_es[:n])
    m = compute_metrics(hyps, gold_va[:n], skip, label)
    print(f"  {label.upper():<10} chrF={m['chrF']:.2f} BLEU={m['BLEU']:.2f} TER={m['TER']:.2f}")
    return m, hyps, skip

In [ ]:
# Baseline
tok_base   = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tok_base.pad_token is None: tok_base.pad_token = tok_base.eos_token
model_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto")
m_base, hyps_base, skip_base = evaluate_model(model_base, tok_base, "BASE")
del model_base; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# SFT
base_sft  = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto")
tok_sft   = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tok_sft.pad_token is None: tok_sft.pad_token = tok_sft.eos_token
model_sft = PeftModel.from_pretrained(base_sft, SFT_HUB_ID).eval()
m_sft, hyps_sft, skip_sft = evaluate_model(model_sft, tok_sft, "SFT")
del model_sft, base_sft; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# GRPO v1
tok_grpo1   = AutoTokenizer.from_pretrained(GRPO_HUB_ID)
if tok_grpo1.pad_token is None: tok_grpo1.pad_token = tok_grpo1.eos_token
model_grpo1 = AutoModelForCausalLM.from_pretrained(GRPO_HUB_ID, quantization_config=bnb_config, device_map="auto").eval()
m_grpo1, hyps_grpo1, skip_grpo1 = evaluate_model(model_grpo1, tok_grpo1, "GRPO")
del model_grpo1; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# GRPO v2
GRPO2_HUB_ID = "guerreropaula/translategemma4b-grpo2-es-va"
tok_grpo2    = AutoTokenizer.from_pretrained(GRPO2_HUB_ID)
if tok_grpo2.pad_token is None: tok_grpo2.pad_token = tok_grpo2.eos_token
model_grpo2  = AutoModelForCausalLM.from_pretrained(GRPO2_HUB_ID, quantization_config=bnb_config, device_map="auto").eval()
m_grpo2, hyps_grpo2, skip_grpo2 = evaluate_model(model_grpo2, tok_grpo2, "GRPO2")
del model_grpo2; torch.cuda.empty_cache(); gc.collect()

---
## 12. Summary Table

In [ ]:
all_metrics = [m_base, m_sft, m_grpo1, m_grpo2]
best = {k: (min if k=="TER" else max)(m[k] for m in all_metrics) for k in ["chrF","BLEU","TER","BLEURT"]}

print("\n" + "="*76)
print(f"  {'Model':<10} {'chrF':>7} {'BLEU':>7} {'TER':>7} {'BLEURT':>8}")
print("="*76)
for m in all_metrics:
    flags = " ◀" if any(m[k]==best[k] for k in ["chrF","BLEU","TER","BLEURT"]) else ""
    print(f"  {m['model']:<10} {m['chrF']:>7.2f} {m['BLEU']:>7.2f} {m['TER']:>7.2f} {m['BLEURT']:>8.4f}{flags}")
print("="*76)

In [ ]:
import json
output = {"dataset":"gplsi/ES-VA_translation_test","n_total":EVAL_N,"results":all_metrics,
          "samples":[{"id":i,"source_es":gold_es[i],"reference_va":gold_va[i],
                      "baseline":hyps_base[i],"sft":hyps_sft[i],
                      "grpo":hyps_grpo1[i],"grpo2":hyps_grpo2[i]} for i in range(EVAL_N)]}
with open("eval_results_1k.json","w",encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("Saved: eval_results_1k.json")